# DeltaJalan — YOLO11 Training (Merged Dataset Clean)

| Kelas | Label |
|-------|-------|
| 0 | lubang |
| 1 | retak_buaya |
| 2 | retak_memanjang |
| 3 | retak_melintang |

**Dataset**: 30,136 images (train 24,106 / valid 3,009 / test 3,021) — 4.1 GB

## Cara Pakai
1. Isi KAGGLE_USERNAME, KAGGLE_KEY, DATASET_SLUG di Cell 3
2. **Run All** — selesai ~1.5 jam (120 epoch)
3. Download zip dari output

> **SageMaker?** Bisa juga. Notebook auto-detect platform.
> **Kaggle?** Tidak perlu install API token manual — kagglehub handle auth via env var.


---
## 1. Environment Detection & Optimasi

In [ ]:
%xmode Minimal

import os, sys, gc, json, shutil, glob, yaml, zipfile, warnings
import locale
locale.getpreferredencoding = lambda: 'UTF-8'
os.environ['ULTRALYTICS_LOG_LEVEL'] = 'WARNING'
warnings.filterwarnings('ignore')

IS_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = '/kaggle/working/' if IS_KAGGLE else os.path.expanduser('~')
os.makedirs(WORK_DIR, exist_ok=True)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Platform: {"Kaggle" if IS_KAGGLE else "SageMaker"} | Device: {DEVICE}')
if DEVICE == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name()} | VRAM: {vram:.1f}GB')

---
## 2. Install & Download Dataset dari Kaggle

In [ ]:
# ========== ISI KREDENSIAL KAGGLE DI SINI ==========
os.environ["KAGGLE_USERNAME"] = "your-username"
os.environ["KAGGLE_KEY"]      = "xxx"
# ====================================================

# ========== ISI SLUG DATASET DI SINI ==========
DATASET_SLUG = "username/dataset-merged-clean"
# ==============================================

assert os.environ["KAGGLE_USERNAME"] != "your-username", "GANTI KAGGLE_USERNAME!"
assert os.environ["KAGGLE_KEY"] != "xxx", "GANTI KAGGLE_KEY!"
assert DATASET_SLUG != "username/dataset-merged-clean", "GANTI DATASET_SLUG!"

# Cek marker: sudah pernah download?
MARKER = os.path.join(WORK_DIR, ".dataset_path")
dataset_path = None
if os.path.exists(MARKER):
    with open(MARKER) as f:
        cached = f.read().strip()
    if cached and os.path.exists(os.path.join(cached, "data.yaml")):
        dataset_path = cached
        print(f"Dataset sudah ada (dari marker): {dataset_path}")
    else:
        print("Marker lama tidak valid. Hapus & download ulang...")
        os.remove(MARKER)

if dataset_path is None:
    !pip install -q kagglehub
    import kagglehub
    print("Downloading dataset...")
    try:
        dataset_path = kagglehub.dataset_download(DATASET_SLUG)
    except Exception as _e:
        print(f"Download gagal: {_e}")
        print("Cek: 1) KAGGLE_USERNAME benar  2) KAGGLE_KEY benar  3) DATASET_SLUG benar  4) Dataset sudah terupload")
        raise
    with open(MARKER, "w") as f:
        f.write(str(dataset_path))
    print(f"Dataset downloaded: {dataset_path}")


---
## 3. Setup data.yaml (absolute path)

In [ ]:
from pathlib import Path

ds_path = Path(dataset_path).resolve()
with open(ds_path / 'data.yaml') as f:
    data_cfg = yaml.safe_load(f)
data_cfg['path'] = str(ds_path)
DATA_YAML = str(ds_path / 'data.yaml')
with open(DATA_YAML, 'w') as f:
    yaml.dump(data_cfg, f)

print(f'data.yaml fixed: {DATA_YAML}')
for split in ['train', 'val', 'test']:
    img_dir = ds_path / data_cfg[split]
    lbl_dir = img_dir.parent / 'labels' / img_dir.name
    imgs = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png')))
    lbls = len(list(lbl_dir.glob('*.txt')))
    print(f'  {split:5s}: {imgs:4d} images, {lbls:4d} labels')
print(f'  Classes ({len(data_cfg["names"])}): {data_cfg["names"]}')

---
## 4. Install Ultralytics & Autobatch

In [ ]:
!pip install -q ultralytics

from ultralytics import YOLO
from ultralytics.utils.autobatch import autobatch

FRESH_MODEL = YOLO("yolo11n.pt")

optimal = autobatch(FRESH_MODEL)
BATCH = max(8, min(int(optimal), 64))
IMGSZ = 640
EPOCHS = 120

print(f"Autobatch result: {optimal:.0f}")
print(f"Using batch: {BATCH} | ImgSize: {IMGSZ} | Epochs: {EPOCHS}")
n_params = sum(p.numel() for p in FRESH_MODEL.model.parameters())
print(f"Params: {n_params:,}")


---
## 5. Training YOLO11n

> **Auto-resume**: Jika notebook crash (kernel mati / tab ditutup), tinggal **Run All** lagi.
> Notebook deteksi otomatis `last.pt` dan resume dari epoch terakhir.
>
> **Checkpoint tiap 20 epoch** — maksimal rugi 20 epoch.


In [ ]:
PROJECT = "deltajalan"
EXP_NAME = "yolo11n_final"
EXP_DIR = os.path.join(WORK_DIR, PROJECT, EXP_NAME)
checkpoint = os.path.join(EXP_DIR, "weights", "last.pt")

if os.path.exists(checkpoint):
    print(f">>> Checkpoint ditemukan: {checkpoint}")
    print(">>> Resume training dari epoch terakhir")
    model = YOLO(checkpoint)
    results = model.train(resume=True)
else:
    print(">>> Start training baru")
    model = FRESH_MODEL

    def memory_cleanup(trainer):
        if (trainer.epoch + 1) % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()
    model.add_callback("on_train_epoch_end", memory_cleanup)

    if DEVICE == "cuda":
        torch.cuda.set_per_process_memory_fraction(0.90)

    results = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        patience=30,
        optimizer="AdamW",
        lr0=0.001, lrf=0.01,
        cos_lr=True,
        mixup=0.2,
        copy_paste=0.3,
        mosaic=1.0,
        close_mosaic=10,
        flipud=0.0,
        fliplr=0.5,
        scale=0.5,
        degrees=10.0,
        workers=2,
        save=True,
        save_period=20,
        val=True,
        amp=True,
        seed=42,
        deterministic=False,
        project=PROJECT,
        name=EXP_NAME,
        exist_ok=True,
        verbose=False,
    )

exp_dir = results.save_dir
print(f"\nTraining selesai! Save dir: {exp_dir}")


---
## 5b. Backup Checkpoint ke S3 (SageMaker Only)

Jika pakai SageMaker, cell ini upload checkpoint ke S3 tiap 10 menit.
Aman meskipun instance JupyterLab dihentikan.


In [ ]:
# ========== ISI BUCKET S3 ==========
S3_BUCKET = "your-s3-bucket"
# ===================================

if IS_KAGGLE:
    print("Bukan SageMaker, lewati S3 backup")
else:
    import subprocess, threading, time as _time
    assert S3_BUCKET != "your-s3-bucket", "GANTI S3_BUCKET!"

    def s3_sync():
        weights_dir = os.path.join(EXP_DIR, "weights")
        while not os.path.exists(weights_dir):
            _time.sleep(30)
        while True:
            for f in ["best.pt", "last.pt"]:
                fp = os.path.join(weights_dir, f)
                if os.path.exists(fp):
                    subprocess.run(
                        ["aws", "s3", "cp", fp, f"s3://{S3_BUCKET}/{f}"],
                        capture_output=True, timeout=60
                    )
            _time.sleep(600)

    t = threading.Thread(target=s3_sync, daemon=True)
    t.start()
    print(f"S3 backup running: s3://{S3_BUCKET}/")


---
## 6. Evaluasi — Test Set

In [ ]:
import os as _os
best_pt = _os.path.join(exp_dir, 'weights', 'best.pt')
last_pt = _os.path.join(exp_dir, 'weights', 'last.pt')

if _os.path.exists(best_pt):
    best_path = best_pt
elif _os.path.exists(last_pt):
    best_path = last_pt
else:
    raise FileNotFoundError(f'Tidak ada model di {exp_dir}/weights/')

best_model = YOLO(best_path)
val = best_model.val(data=DATA_YAML, split='test', imgsz=IMGSZ, batch=BATCH, verbose=False)

print('='*60)
print('HASIL VALIDASI — TEST SET')
print('='*60)
print(f'mAP50-95:  {val.box.map:.4f}')
print(f'mAP50:     {val.box.map50:.4f}')
print(f'Precision: {val.box.mp:.4f}')
print(f'Recall:    {val.box.mr:.4f}')
print()
print('Per-Class:')
for i, name in best_model.names.items():
    p = val.box.p[i]
    r = val.box.r[i]
    f1 = 2 * p * r / (p + r + 1e-10)
    ap50 = val.box.ap50[i]
    ap5095 = val.box.ap[i].mean()
    print(f'  {name:22s}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}  mAP50={ap50:.4f}  mAP50-95={ap5095:.4f}')

---
## 7. Export ONNX

In [ ]:
best_model.export(format='onnx', imgsz=IMGSZ, dynamic=True)
print('ONNX export selesai!')

!ls -lh {_os.path.join(exp_dir, 'weights')}/

---
## 8. Package & Download

In [ ]:
weights = _os.path.join(exp_dir, 'weights')
output_zip = _os.path.join(WORK_DIR, 'deltajalan_model_final.zip')

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in ['best.pt', 'last.pt', 'best.onnx']:
        fp = _os.path.join(weights, f)
        if _os.path.exists(fp):
            zf.write(fp, f'weights/{f}')
    log = _os.path.join(exp_dir, 'results.csv')
    if _os.path.exists(log):
        zf.write(log, 'results.csv')

print(f'Package: {output_zip}')
print(f'Size:    {_os.path.getsize(output_zip)/1e6:.1f} MB')